# Drilling Engineering — Worked Examples

This notebook demonstrates practical applications of the `drilling.py` module.
All calculations are referenced to industry-standard textbooks.

**Contents:**
1. Well control — Kill mud weight & pressure schedule
2. Hydraulics — ECD and annular pressure loss
3. Casing design — Burst, collapse, and tension safety factors
4. Directional drilling — Survey computation and trajectory plot
5. Pore pressure — Eaton fracture gradient and mud weight window
6. ROP optimization — Cost per foot analysis

**Reference:** Bourgoyne, A.T. et al. (1986). *Applied Drilling Engineering*. SPE Textbook Vol. 2.

In [ ]:
import sys
sys.path.insert(0, '../src')
import drilling as drl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

print('Drilling Engineering module loaded.')

---
## 1. Well Control

**Scenario:** Kick detected while drilling at 9,500 ft TVD.
- Current MW = 10.5 ppg
- SIDPP = 350 psi | SICP = 580 psi
- Pit gain = 12 bbl | SCR pressure = 650 psi


In [ ]:
# Input parameters
mw       = 10.5   # ppg
sidpp    = 350    # psi
sicp     = 580    # psi
tvd      = 9500   # ft
pit_gain = 12.0   # bbl
scr_psi  = 650    # psi
ann_cap  = 0.0775 # bbl/ft

# Calculations
kmw     = drl.kill_mud_weight(mw, sidpp, tvd)
icp     = drl.initial_circulating_pressure(sidpp, scr_psi)
fcp     = drl.final_circulating_pressure(scr_psi, kmw, mw)
fp      = drl.formation_pressure(mw, tvd, sidpp)
influx  = drl.influx_type(mw, sicp, sidpp, ann_cap, pit_gain)

print(f'Kill Mud Weight (KMW)   : {kmw:.2f} ppg')
print(f'Formation Pressure      : {fp:,.0f} psi  ({fp/tvd/0.052:.2f} ppg EMW)')
print(f'Initial Circ. Pressure  : {icp:,.0f} psi')
print(f'Final Circ. Pressure    : {fcp:,.0f} psi')
print(f'Influx type estimate    : {influx.upper()}')
print(f'Influx column height    : {pit_gain/ann_cap:.1f} ft')

In [ ]:
# W&W pressure schedule
dp_capacity = 0.01776  # bbl/ft
dp_vol = dp_capacity * tvd
strokes = np.linspace(0, dp_vol / 0.1, 50)
pressures = np.linspace(icp, fcp, len(strokes))

plt.figure(figsize=(9, 4))
plt.plot(strokes, pressures, 'b-', linewidth=2, label='SIDPP target')
plt.axhline(y=icp, color='r', linestyle='--', alpha=0.6, label=f'ICP = {icp:.0f} psi')
plt.axhline(y=fcp, color='g', linestyle='--', alpha=0.6, label=f'FCP = {fcp:.0f} psi')
plt.xlabel('Strokes')
plt.ylabel('Drill Pipe Pressure (psi)')
plt.title('Wait & Weight Pressure Schedule')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Hydraulics & ECD

In [ ]:
# Mud properties
mw_h, pv, yp = 11.0, 18, 12
q_gpm = 420
hole, dp_od, dc_od = 8.5, 5.0, 6.5
dc_len, dp_len = 600, 8900
tvd_h = 9500

al_dc = drl.annular_pressure_loss_bingham(mw_h, pv, yp, q_gpm, hole, dc_od, dc_len)
al_dp = drl.annular_pressure_loss_bingham(mw_h, pv, yp, q_gpm, hole, dp_od, dp_len)
total_al = al_dc + al_dp
ecd_val  = drl.ecd(mw_h, total_al, tvd_h)
jv       = drl.jet_velocity(q_gpm, [12, 12, 13])
impact   = drl.impact_force(mw_h, q_gpm, jv)

print(f'Ann. loss — DC section  : {al_dc:.1f} psi')
print(f'Ann. loss — DP section  : {al_dp:.1f} psi')
print(f'Total annular loss       : {total_al:.1f} psi')
print(f'ECD                      : {ecd_val:.3f} ppg (+{ecd_val-mw_h:.3f} ppg over MW)')
print(f'Jet velocity             : {jv:.0f} ft/s')
print(f'Bit impact force         : {impact:.0f} lbf')

# ECD vs flow rate sensitivity
q_range = np.arange(100, 700, 10)
ecd_curve = []
for q_ in q_range:
    al = drl.annular_pressure_loss_bingham(mw_h, pv, yp, q_, hole, dp_od, dp_len) + \
         drl.annular_pressure_loss_bingham(mw_h, pv, yp, q_, hole, dc_od, dc_len)
    ecd_curve.append(drl.ecd(mw_h, al, tvd_h))

plt.figure(figsize=(9, 4))
plt.plot(q_range, ecd_curve, 'purple', linewidth=2)
plt.axhline(y=mw_h, color='b', linestyle=':', label=f'Static MW = {mw_h} ppg')
plt.axvline(x=q_gpm, color='r', linestyle='--', alpha=0.6, label=f'Q = {q_gpm} gpm')
plt.xlabel('Flow Rate (gpm)')
plt.ylabel('ECD (ppg)')
plt.title('ECD Sensitivity to Flow Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Directional Drilling — Minimum Curvature Survey

In [ ]:
# Survey data for a typical J-type well
survey = [
    (0,     0,    0),
    (2000,  0,    0),
    (4000,  15,  45),
    (5500,  35,  45),
    (7000,  55,  45),
    (8500,  75,  45),
    (10000, 89,  45),
    (11500, 90,  45),
]

mds  = [s[0] for s in survey]
incs = [s[1] for s in survey]
azis = [s[2] for s in survey]

tvds = [0.0]; norths = [0.0]; easts = [0.0]; dls_vals = [0.0]
for i in range(1, len(mds)):
    dt, dn, de = drl.minimum_curvature(mds[i-1], mds[i], incs[i-1], azis[i-1], incs[i], azis[i])
    tvds.append(tvds[-1] + dt)
    norths.append(norths[-1] + dn)
    easts.append(easts[-1] + de)
    dls = drl.dogleg_severity(incs[i-1], azis[i-1], incs[i], azis[i], mds[i]-mds[i-1])
    dls_vals.append(dls)

df_survey = pd.DataFrame({
    'MD (ft)':       [round(x) for x in mds],
    'Inc (°)':       incs,
    'Azi (°)':       azis,
    'TVD (ft)':      [round(x, 1) for x in tvds],
    'North (ft)':    [round(x, 1) for x in norths],
    'East (ft)':     [round(x, 1) for x in easts],
    'DLS (°/100ft)': [round(x, 2) for x in dls_vals],
})
print(df_survey.to_string(index=False))

# 2D vertical section plot
departure = [math.sqrt(n**2 + e**2) for n, e in zip(norths, easts)]
plt.figure(figsize=(8, 6))
plt.plot(departure, tvds, 'b-o', linewidth=2, markersize=5)
plt.gca().invert_yaxis()
plt.xlabel('Horizontal Departure (ft)')
plt.ylabel('TVD (ft)')
plt.title('Well Trajectory — Vertical Section')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Pore Pressure & Fracture Gradient — Eaton Method

In [ ]:
depths = np.arange(500, 14000, 200)  # ft
obg    = 0.9  # psi/ft overburden
nu     = 0.25  # Poisson's ratio

# Pore pressure gradient — normal with overpressure below 8000 ft
pp_grads = np.where(depths > 10000, 0.60,
           np.where(depths > 7500, 0.52, 0.433))

fg_grads = np.array([drl.eaton_fracture_gradient(obg, ppg, nu) for ppg in pp_grads])

fig, ax = plt.subplots(figsize=(6, 9))
ax.plot(pp_grads, depths, 'r-', linewidth=2, label='Pore pressure gradient')
ax.plot(fg_grads, depths, 'g--', linewidth=2, label='Fracture gradient (Eaton)')
ax.fill_betweenx(depths, pp_grads + 0.05, fg_grads - 0.05,
                  alpha=0.12, color='blue', label='Safe MW window')
ax.invert_yaxis()
ax.set_xlabel('Pressure gradient (psi/ft)')
ax.set_ylabel('Depth (ft)')
ax.set_title('Pore Pressure & Fracture Gradient\n(Eaton, 1969)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. ROP Optimization — Cost Per Foot Analysis

In [ ]:
# Base parameters
bit_cost  = 18000    # USD
rig_rate  = 2500     # USD/hr
trip_time = 8.0      # hr
footage   = 1500     # ft
rpm       = 100
bs        = 8.5      # in
a1, a2    = 0.008, 1.2

wob_range = np.arange(5, 55, 2)  # klbf
rop_vals  = [drl.bingham_rop_model(w*1000, rpm, bs, a1, a2) for w in wob_range]
cpf_vals  = []
for rop_i in rop_vals:
    dt = footage / rop_i if rop_i > 0 else 9999
    cpf_vals.append(drl.cost_per_foot(bit_cost, rig_rate, dt, trip_time, footage))

opt_idx = np.argmin(cpf_vals)
print(f'Optimal WOB  : {wob_range[opt_idx]} klbf')
print(f'Min CPF      : ${cpf_vals[opt_idx]:,.2f}/ft')
print(f'ROP at opt   : {rop_vals[opt_idx]:.1f} ft/hr')

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.plot(wob_range, rop_vals, 'b-', linewidth=2, label='ROP (ft/hr)')
ax2.plot(wob_range, cpf_vals, 'r-', linewidth=2, label='CPF ($/ft)')
ax1.axvline(x=wob_range[opt_idx], color='g', linestyle='--',
            label=f'Optimal WOB = {wob_range[opt_idx]} klbf')
ax1.set_xlabel('WOB (klbf)')
ax1.set_ylabel('ROP (ft/hr)', color='b')
ax2.set_ylabel('Cost per Foot ($/ft)', color='r')
ax1.set_title('ROP and Cost-per-Foot vs Weight on Bit')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()